**Variables, Domains, and Constraints**

In [12]:
variables = ['India', 'Pakistan', 'Bangladesh', 'Nepal', 'Bhutan', 'SriLanka', 'Afghanistan']

domains = {v: ['Red', 'Green', 'Blue'] for v in variables}

constraints = [
    ('India',       'Pakistan'),
    ('India',       'Bangladesh'),
    ('India',       'Nepal'),
    ('India',       'Bhutan'),
    ('India',       'Afghanistan'),
    ('Pakistan',    'Afghanistan'),
    ('Nepal',       'Bhutan'),
]

neighbors = {v: set() for v in variables}
for (xi, xj) in constraints:
    neighbors[xi].add(xj)
    neighbors[xj].add(xi)

**Implement the REVISE Function**

In [13]:
def revise(domains, xi, xj):
    revised = False
    removed = []

    for x in domains[xi][:]:
        has_support = any(x != y for y in domains[xj])

        if not has_support:
            domains[xi].remove(x)
            removed.append(x)
            revised = True

    return revised, removed



**Testing the Revised Function**

In [14]:
print('REVISE Function — Sanity Test')
test_domains = {'A': ['Red'], 'B': ['Red']}
rev, rem = revise(test_domains, 'A', 'B')
print(f'Test: A=[Red], B=[Red]')
print(f'  → Revised: {rev}, Removed from A: {rem}')
print(f'  → Domain(A) after REVISE: {test_domains["A"]}')
print()
test_domains2 = {'A': ['Red', 'Green'], 'B': ['Red']}
rev2, rem2 = revise(test_domains2, 'A', 'B')
print(f'Test: A=[Red, Green], B=[Red]')
print(f'  → Revised: {rev2}, Removed from A: {rem2}')
print(f'  → Domain(A) after REVISE: {test_domains2["A"]}')
print()
print('✅ REVISE function works correctly.')

REVISE Function — Sanity Test
Test: A=[Red], B=[Red]
  → Revised: True, Removed from A: ['Red']
  → Domain(A) after REVISE: []

Test: A=[Red, Green], B=[Red]
  → Revised: True, Removed from A: ['Red']
  → Domain(A) after REVISE: ['Green']

✅ REVISE function works correctly.


**the AC-3 Algorithm**

In [15]:
from collections import deque
import copy

def ac3(domains, constraints, neighbors, verbose=True):
    queue = deque()
    for (xi, xj) in constraints:
        queue.append((xi, xj))
        queue.append((xj, xi))

    step = 0
    if verbose:
        print('=' * 60)
        print('           AC-3 ALGORITHM — Step-by-Step Trace')
        print('=' * 60)
        print(f'Initial queue size: {len(queue)} arcs')
        print()

    while queue:
        (xi, xj) = queue.popleft()
        step += 1

        before = domains[xi][:]           
        revised, removed = revise(domains, xi, xj)

        if verbose:
            status = f'  Removed {removed}' if revised else '  No change'
            print(f'Step {step:>2}: Arc ({xi} → {xj})')
            print(f'         domain({xi}) before : {before}')
            print(f'         domain({xi}) after  : {domains[xi]}')
            print(f'         {status}')

        if revised:
            if len(domains[xi]) == 0:
                if verbose:
                    print(f'\n❌ FAILURE: domain({xi}) is empty — no solution possible!')
                return False
            re_added = []
            for xk in neighbors[xi]:
                if xk != xj:
                    queue.append((xk, xi))
                    re_added.append(xk)
            if verbose and re_added:
                print(f'         Re-added arcs: {[(k, xi) for k in re_added]}')

        if verbose:
            print()

    if verbose:
        print('=' * 60)
        print('✅ AC-3 completed — arc consistency enforced!')
        print('=' * 60)

    return True


print('AC-3 function defined successfully.')

AC-3 function defined successfully.


**Run AC-3**

In [16]:
import copy
working_domains = copy.deepcopy(domains)

result = ac3(working_domains, constraints, neighbors, verbose=True)

           AC-3 ALGORITHM — Step-by-Step Trace
Initial queue size: 14 arcs

Step  1: Arc (India → Pakistan)
         domain(India) before : ['Red', 'Green', 'Blue']
         domain(India) after  : ['Red', 'Green', 'Blue']
           No change

Step  2: Arc (Pakistan → India)
         domain(Pakistan) before : ['Red', 'Green', 'Blue']
         domain(Pakistan) after  : ['Red', 'Green', 'Blue']
           No change

Step  3: Arc (India → Bangladesh)
         domain(India) before : ['Red', 'Green', 'Blue']
         domain(India) after  : ['Red', 'Green', 'Blue']
           No change

Step  4: Arc (Bangladesh → India)
         domain(Bangladesh) before : ['Red', 'Green', 'Blue']
         domain(Bangladesh) after  : ['Red', 'Green', 'Blue']
           No change

Step  5: Arc (India → Nepal)
         domain(India) before : ['Red', 'Green', 'Blue']
         domain(India) after  : ['Red', 'Green', 'Blue']
           No change

Step  6: Arc (Nepal → India)
         domain(Nepal) before : ['Red'

**Reduced Domains**

In [19]:
print('         FINAL DOMAINS AFTER AC-3')
print(f'{"Region":<10} {"Initial Domain":<30} {"Reduced Domain"}')

for var in variables:
    initial = ['Red', 'Green', 'Blue']
    final   = working_domains[var]
    changed = '  ← reduced' if len(final) < len(initial) else ''
    print(f'{var:<10} {str(initial):<30} {final}{changed}')


print()

if result:
    print('✅ AC-3 succeeded — no domain became empty.')
    print()
    # Highlight variables whose domain was reduced
    reduced = [v for v in variables if len(working_domains[v]) < 3]
    if reduced:
        print(f'Variables with reduced domains: {reduced}')
    else:
        print('No domains were reduced — all remain {Red, Green, Blue}.')
else:
    print('❌ AC-3 failed — the problem has no solution.')

         FINAL DOMAINS AFTER AC-3
Region     Initial Domain                 Reduced Domain
India      ['Red', 'Green', 'Blue']       ['Red', 'Green', 'Blue']
Pakistan   ['Red', 'Green', 'Blue']       ['Red', 'Green', 'Blue']
Bangladesh ['Red', 'Green', 'Blue']       ['Red', 'Green', 'Blue']
Nepal      ['Red', 'Green', 'Blue']       ['Red', 'Green', 'Blue']
Bhutan     ['Red', 'Green', 'Blue']       ['Red', 'Green', 'Blue']
SriLanka   ['Red', 'Green', 'Blue']       ['Red', 'Green', 'Blue']
Afghanistan ['Red', 'Green', 'Blue']       ['Red', 'Green', 'Blue']

✅ AC-3 succeeded — no domain became empty.

No domains were reduced — all remain {Red, Green, Blue}.
